In [1]:
import tiktoken
import torch
from previous_chapters import GPTModel
from gpt_train import train_model_simple
import os
import requests
from previous_chapters import create_dataloader_v1

C:\Users\Qiuyw\.conda\envs\LLMBuilding\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [6]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,   # 词汇表大小
    "context_length": 256, # 缩小的上下文长度 (原来: 1024)
    "emb_dim": 768,        # 嵌入维度
    "n_heads": 12,         # 注意力头数量
    "n_layers": 12,        # 层数
    "drop_rate": 0.1,      # Dropout 率
    "qkv_bias": False      # 查询-键-值偏置
}

model = GPTModel(GPT_CONFIG_124M)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

model.load_state_dict(torch.load("model.pth", map_location=device, weights_only=True))
model.eval();

In [7]:
file_path = "the-verdict.txt"
with open(file_path, "r", encoding="utf-8") as file:
        text_data = file.read()

In [8]:
train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]


torch.manual_seed(123)

train_loader = create_dataloader_v1(
    train_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)

val_loader = create_dataloader_v1(
    val_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

In [9]:
num_epochs = 1
tokenizer = tiktoken.get_encoding("gpt2")
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)

train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context="Every effort moves you", tokenizer=tokenizer
)

Ep 1 (Step 000000): Train loss 1.895, Val loss 7.360
Ep 1 (Step 000005): Train loss 0.835, Val loss 6.521
Every effort moves you?" "Oh, my hostess was "interesting--as such--had not existed till nearly a year after Jack's resolve had been taken. It might be that he had married her--since he had been Rome: "Be dissatisfied with his
